# 지도학습 및 출력 증류 실행

ViT-B 증류, ViT-S 지도학습 및 ViT-S 추가 증류를 동일한 실행 단위로 구성한다. 추가 증류는 완료된 ViT-S 전체 분할 가중치를 초기값으로 사용한다. 데이터와 가중치는 `local_paths.json`에 지정한다. 기본 상태에서는 구성을 조회하며, 실제 실행은 아래 설정으로 활성화한다.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Open this notebook from the repository'
sys.path.insert(0, str(ROOT / 'src'))
import torch
torch.set_num_threads(2)
torch.manual_seed(20261112)
print('Repository', ROOT.name, '| PyTorch', torch.__version__)

## 1. 실험 설정

In [ ]:
import json
RECIPE = 'vits-distillation'  # also: 'vits-supervised', 'vitb-distillation'
DEVICE = 'cuda'
LOAD_PRIVATE_INPUTS = False
RUN_ONE_EPOCH = False
RUN_ALL_EPOCHS = False
config = json.loads((ROOT / f'configs/{RECIPE}.json').read_text())
# Keep notebook experiments separate from an unattended run.
run_dir = ROOT / 'runs' / (config['name'] + '-notebook')
print(config)
print('Run directory', run_dir.name)

## 2. 데이터 분할 및 이미지·라벨 대응 검사

In [ ]:
from cpdino_lab.data import discover_splits, make_batch, read_local_paths
from cpdino_lab.training import dataset_fingerprint
if LOAD_PRIVATE_INPUTS:
    paths = read_local_paths(ROOT / 'local_paths.json')
    splits = discover_splits(paths['dataset_root'], training_split=config.get('training_split'))
    dataset_hash = dataset_fingerprint(splits)
    print({key:len(value) for key,value in splits.items()})
else:
    print('Set local_paths.json and LOAD_PRIVATE_INPUTS to load the selected dataset')

## 3. 모델 구성 및 가중치 초기화

In [ ]:
from cpdino_lab.training import prepare_models, EpochStore, train_epoch
from cpdino_lab.checkpoints import save_checkpoint, restore_checkpoint
if LOAD_PRIVATE_INPUTS:
    student, teacher, optimizer, provenance = prepare_models(config, paths, DEVICE)
    identity = {'recipe':config, **provenance, 'dataset_sha256':dataset_hash}
    store = EpochStore(run_dir, identity, student, optimizer)
    store.resume()
    print('Ready', config['name'], 'at epoch', store.epoch)
    print('Initialization', provenance['student']['scope'], '| teacher', config['teacher'])

## 4. 단일 batch 학습 연산 검증

첫 번째 학습 이미지로 파라미터 갱신을 수행한다. 점검 이전 상태를 저장하고 이후 복원하여 본 학습의 초기 상태를 유지한다.

In [ ]:
from cpdino_lab.engine import train_step, evaluate
if LOAD_PRIVATE_INPUTS:
    save_checkpoint(run_dir/'inspection.pt', student, optimizer, store.epoch, identity,
                    training_state={'best_epoch':store.best_epoch,'best_loss':store.best_loss})
    try:
        x, y = make_batch(splits['train'], [0], seed=42, device=DEVICE, tile_size=config['tile_size'])
        loss = train_step(student, optimizer, x, y, teacher, config['teacher_weight'])
        print('Batch shape', tuple(x.shape), '| loss', loss)
        assert teacher is None or all(p.grad is None for p in teacher.parameters())
    finally:
        restore_checkpoint(run_dir/'inspection.pt', student, optimizer, identity)

## 5. 한 epoch의 학습, 검증 및 체크포인트 선정

`RUN_ONE_EPOCH=True`에서 다음 한 epoch를 수행한다. 첫 epoch와 이후 10 epochs마다 지도 손실로 검증하며, 최저 검증 손실의 `best.pt`와 재개용 `last.pt`를 저장한다. 최적 epoch 정보도 재개 시 복원한다.

In [ ]:
if LOAD_PRIVATE_INPUTS and RUN_ONE_EPOCH:
    row = train_epoch(student, optimizer, teacher, splits, config, store, DEVICE)
    print(row)
else:
    print('One epoch requires LOAD_PRIVATE_INPUTS=True and RUN_ONE_EPOCH=True')

## 6. 설정된 학습 길이까지 반복 실행

`RUN_ALL_EPOCHS=True`에서 동일한 epoch 단위를 반복한다. ViT-S 증류의 500 epochs는 선행 지도학습 500 epochs 이후의 추가 학습 길이이다. 한 batch 점검과 한 epoch 실행 이후에도 남은 epoch만 수행한다.

In [ ]:
if LOAD_PRIVATE_INPUTS and RUN_ALL_EPOCHS:
    while store.epoch < config['epochs']:
        print(train_epoch(student, optimizer, teacher, splits, config, store, DEVICE))
else:
    print('Full training is disabled in inspection mode')

## 7. 독립 test 평가

학습 완료 후 선택된 `best.pt`에 대해 동일한 64장 평가 설정을 적용한다. 평가 결과의 원본에는 이미지 식별자가 포함되므로 run 디렉터리에 보관하고, 공개 문서에는 `public_summary`가 반환하는 집계값을 사용한다.

In [ ]:
from cpdino_lab.evaluation import evaluate_checkpoint, public_summary
RUN_TEST_EVALUATION = False
if LOAD_PRIVATE_INPUTS and RUN_TEST_EVALUATION:
    assert store.epoch == config['epochs'], 'Complete the configured training budget before final evaluation'
    result = evaluate_checkpoint(run_dir/'best.pt', splits['test'], paths['vitb_model']['path'],
                                 run_dir/'evaluation', backbone=config['student'], model_id=config['name'])
    print(public_summary(result))
else:
    print('Final test evaluation is disabled in inspection mode')